In [ ]:
import requests
import pandas as pd
import json

GAMMA_URL = "https://gamma-api.polymarket.com/markets"
CLOB_URL = "https://clob.polymarket.com"

def fetch_markets(limit=1000, offset=0):
    params = {
        "limit": limit,
        "offset": offset,
        "closed": "true"
    }
    response = requests.get(GAMMA_URL, params=params)
    response.raise_for_status()
    return response.json()

def is_binary_market(market):
    outcomes = market.get("outcomes", [])
    outcomes = json.loads(outcomes)
    return len(outcomes) == 2

def is_resolved(market):
    return market.get("closed") is True

def has_min_duration(market, min_days=30):
    try:
        start = pd.to_datetime(market.get("startDate"))
        end = pd.to_datetime(market.get("endDate"))
        return (end - start) >= pd.Timedelta(days=min_days)
    except:
        return False
    
def fetch_price_history(token_id):
    url = f"{CLOB_URL}/prices-history"

    params = {
        "market": token_id,
        "interval": "1d",
        "fidelity": 1440
    }

    r = requests.get(url, params=params, timeout=10)

    if r.status_code != 200:
        print("Request failed:", r.status_code, r.text)


    all_data = r.json().get("history", [])
    print(r.json)
    print(all_data)

    if not all_data:
        return None
    
    df = pd.DataFrame(all_data)
    return df

    # print("df", df.head())
    
def is_valid_market(market):
    if "clobTokenIds" not in market:
        return False
    
    return (
        is_binary_market(market) and
        is_resolved(market) and
        (market.get("volumeNum", 0) > 0) and
        has_min_duration(market)
    )

In [23]:
offset = 0
all_markets = []
while len(all_markets) < 1000:
    markets = fetch_markets(offset=offset)
    print("markets", len(markets))
    print("all markets", len(all_markets))
    offset += 1000
    filtered = [m for m in markets if is_valid_market(m)]
    all_markets += filtered

print("All markets:", len(all_markets))

markets 1000
all markets 0
markets 1000
all markets 57
markets 1000
all markets 117
markets 1000
all markets 337
markets 1000
all markets 464
markets 1000
all markets 541
markets 1000
all markets 623
markets 1000
all markets 665
markets 1000
all markets 911
All markets: 1271


In [30]:
good_token = 49530104934124489931642936878451731683144971145438325491355482494203203593939  

thing = fetch_price_history(good_token)
print(thing)

<bound method Response.json of <Response [200]>>
[]
None


In [ ]:
from concurrent.futures import ThreadPoolExecutor

def process_market(market):
    tokens = json.loads(market["clobTokenIds"])
    df = fetch_price_history(tokens[0])

    if df is not None and not df.empty:
        return market
    return None


with ThreadPoolExecutor(max_workers=10) as executor:
    results = list(executor.map(process_market, all_markets))

filtered_markets = [r for r in results if r is not None]

num = 0
filtered_markets = []
for market in all_markets:
    tokens = json.loads(market["clobTokenIds"])
    df = fetch_price_history(tokens[0])

    if not df.empty:
        num += 1
        filtered_markets.append(market)
        print(num)

print("Filtered markets:", len(filtered_markets))

<bound method Response.json of <Response [200]>>
[]
None


In [ ]:
BASE_URL = "https://data-api.polymarket.com"

def get_recent_trades(limit=20):
    url = f"{BASE_URL}/trades"
    params = {"limit": limit}

    r = requests.get(url, params=params, timeout=10)
    r.raise_for_status()
    return r.json()

def main():
    trades = get_recent_trades(limit=20)

    df = pd.DataFrame(trades)
    print("Columns:", df.columns.tolist())
    print()
    print(df.head())

if __name__ == "__main__":
    main()

In [ ]:
def fetch_price_history(token_id):
    url = f"{CLOB_URL}/prices-history"

    params = {
        "market": token_id,
        "interval": "1d"
    }

    r = requests.get(url, params=params, timeout=10)

    if r.status_code != 200:
        print("Request failed:", r.status_code, r.text)

    all_data = r.json()

    if not all_data:
        return None
    
    df = pd.DataFrame(all_data)

    print("df", df.head())

if __name__ == "__main__":
    print(json.loads(filtered[0]["clobTokenIds"])[0])
    fetch_price_history(json.loads(filtered[0]["clobTokenIds"])[0])

In [ ]:
def get_price_at_horizon(price_df, resolution_time, days_before):
    target_time = resolution_time - pd.Timedelta(days=days_before)

    df_before = price_df[price_df["timestamp"] <= target_time]

    if df_before.empty:
        return None

    return df_before.iloc[-1]["price"]

def build_market_features(price_df, resolution_time):
    return {
        "p_30d": get_price_at_horizon(price_df, resolution_time, 30),
        "p_7d": get_price_at_horizon(price_df, resolution_time, 7),
        "p_1d": get_price_at_horizon(price_df, resolution_time, 1),
    }

def get_outcome(market):
    return int(market["outcome"] == "Yes")  # adjust if needed

In [ ]:
dataset = []

for market in filtered:
    try:
        tokens = json.loads(market["clobTokenIds"])
        token_id = tokens[0]

        price_df = fetch_price_history(token_id)
        if price_df is None:
            continue

        resolution_time = pd.to_datetime(market["endDate"])
        outcome = get_outcome(market)

        features = build_market_features(price_df, resolution_time)

        row = {
            "market_id": market["id"],
            "outcome": outcome,
            **features
        }

        dataset.append(row)

    except Exception as e:
        continue

df_final = pd.DataFrame(dataset)
print(df_final.head())